# LeetCode 3256 — Maximum Value Sum by Placing Three Rooks I bằng Python

- Link: https://leetcode.com/problems/maximum-value-sum-by-placing-three-rooks-i/
- Độ khó: **Hard**
- Chủ đề: **Chỉ cần top-3 mỗi hàng** (hai quân còn lại chặn tối đa hai cột) ⇒ duyệt **mọi bộ ba hàng** × `3³` lựa chọn: `O(m³ · 27)` với `m ≤ 100`; so sánh với kỹ thuật tiền tố/hậu tố của bản II

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maximumValueSum(self, board: List[List[int]]) -> int:
```

## 1. Mục tiêu học

1. Ba quân xe không ăn nhau ⇔ ba ô ở **ba hàng khác nhau** và **ba cột khác nhau**.
2. Nhận xét then chốt: trong một lời giải tối ưu, có thể giả sử mỗi quân xe nằm trong **top-3 ô lớn nhất của hàng nó** — hai quân kia chặn tối đa 2 cột, nên luôn còn một ô trong top-3 dùng được và không nhỏ hơn.
3. Với `m ≤ 100`: duyệt mọi bộ ba hàng `C(100, 3) ≈ 1.6·10^5`, mỗi bộ thử `≤ 27` tổ hợp ô ⇒ `≈ 4.4·10^6` phép — đủ nhanh.
4. Biết thêm cách `O(m · n · 9)` (tiền tố/hậu tố top-3, cố định hàng giữa) dùng cho bản II.
5. Kiểm chứng bằng vét cạn mọi bộ ba ô.

## 2. Đề bài & ví dụ

Cho bàn cờ `m × n` với giá trị `board[i][j]` (có thể âm). Đặt **ba** quân xe sao cho không quân nào ăn quân nào (không chung hàng, không chung cột). Trả về tổng giá trị lớn nhất của ba ô.

**Ví dụ 1:** `[[-3,1,1,1],[-3,1,-3,1],[-3,2,1,1]]` → `4`.
**Ví dụ 2:** `[[1,2,3],[4,5,6],[7,8,9]]` → `15`.
**Ví dụ 3:** `[[1,1,1],[1,1,1],[1,1,1]]` → `3`.

**Ràng buộc:** `3 ≤ m, n ≤ 100`; `−10^9 ≤ board[i][j] ≤ 10^9`.

(Bản II — LC 3257 — tăng lên `500 × 500`, buộc dùng tiền tố/hậu tố.)

In [1]:
from typing import List
from itertools import combinations, permutations

examples = [([[-3, 1, 1, 1], [-3, 1, -3, 1], [-3, 2, 1, 1]], 4), ([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 15),
            ([[1, 1, 1], [1, 1, 1], [1, 1, 1]], 3)]
for b, exp in examples:
    print(b, '-> kỳ vọng', exp)

[[-3, 1, 1, 1], [-3, 1, -3, 1], [-3, 2, 1, 1]] -> kỳ vọng 4
[[1, 2, 3], [4, 5, 6], [7, 8, 9]] -> kỳ vọng 15
[[1, 1, 1], [1, 1, 1], [1, 1, 1]] -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1. Top-3 của một hàng

In [2]:
def top3_row(row):
    return sorted(((v, j) for j, v in enumerate(row)), reverse=True)[:3]

print(top3_row([-3, 1, -3, 1]))

[(1, 3), (1, 1), (-3, 2)]


### 3.2. Gộp top-3 **theo cột phân biệt**

Phần "trên" gồm nhiều hàng; quân trên chỉ có **một**, nên ta cần 3 ô tốt nhất với **cột khác nhau** (đủ để tránh 2 cột bị quân giữa và quân dưới chiếm). Gộp danh sách mới vào danh sách hiện có, giữ mỗi cột một ô tốt nhất, rồi lấy 3 ô lớn nhất.

In [3]:
def merge_top3(cur, cand):
    best = {}
    for v, j in cur + cand:
        if j not in best or v > best[j]:
            best[j] = v
    return sorted(((v, j) for j, v in best.items()), reverse=True)[:3]

print(merge_top3([(5, 0), (4, 1)], [(6, 0), (3, 2), (2, 3)]))

[(6, 0), (4, 1), (3, 2)]


### 3.3. Vì sao top-3 là đủ?

Xét quân `A` trong lời giải tối ưu, hai quân còn lại chiếm 2 cột. Trong top-3 của tập ứng viên cho `A` (cột phân biệt) có ít nhất một ô không ở 2 cột đó, giá trị `≥` ô bất kỳ ngoài top-3 ⇒ thay `A` bằng ô đó không làm giảm tổng, vẫn hợp lệ (hàng vẫn đúng phần).

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1: tham lam "mỗi hàng lấy ô lớn nhất" — hàng 2 lấy `2` (cột 1), hàng 0 và hàng 1 có nhiều số `1`: hàng 0 lấy cột 2, hàng 1 lấy cột 3 ⇒ `1 + 1 + 2 = 4` ✓. Nhưng nếu hai hàng có ô lớn nhất **cùng cột**, phải nhường — top-3 cho đủ lựa chọn thay thế.

## 5. Cách 1 — Brute Force: mọi bộ ba hàng và ba cột

In [4]:
def brute(board):
    m, n = len(board), len(board[0])
    best = None
    for rows in combinations(range(m), 3):
        for cols in permutations(range(n), 3):
            s = sum(board[r][c] for r, c in zip(rows, cols))
            if best is None or s > best:
                best = s
    return best

for b, exp in examples:
    print(brute(b), '| kỳ vọng', exp)

4 | kỳ vọng 4
15 | kỳ vọng 15
3 | kỳ vọng 3


**Độ phức tạp:** `O(C(m, 3) · n³)` — chỉ cho bàn nhỏ.

## 6. Ý tưởng thuật toán / bước đột phá

```
tops[i] = top-3 (v, cột) của hàng i
for i < j < k (ba hàng):
    for (a, ca) ∈ tops[i], (b, cb) ∈ tops[j], (c, cc) ∈ tops[k]:
        nếu ca, cb, cc đôi một khác nhau: ans = max(ans, a + b + c)
```

`O(m n log n + C(m, 3) · 27)` ≈ `4.4·10^6` với `m = 100`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Top-3 mỗi hàng + mọi bộ ba hàng (`O(m³ · 27)`) — ý tưởng của lời giải nộp

In [5]:
def row_triples(board):
    m = len(board)
    tops = [top3_row(r) for r in board]
    best = None
    for i, j, k in combinations(range(m), 3):
        for a, ca in tops[i]:
            for b, cb in tops[j]:
                if cb == ca:
                    continue
                for c, cc in tops[k]:
                    if cc != ca and cc != cb:
                        s = a + b + c
                        if best is None or s > best:
                            best = s
    return best

for b, exp in examples:
    print(row_triples(b), '| kỳ vọng', exp)

4 | kỳ vọng 4
15 | kỳ vọng 15
3 | kỳ vọng 3


Đủ nhanh khi `m ≤ 100` (bài này); với `m = 500` (LC 3257) quá chậm (`2·10^7 · 27`).

### 7.2. Cách 3 — Tiền tố / hậu tố top-3 + hàng giữa (cho bản II)

In [6]:
def prefix_suffix(board):
    m, n = len(board), len(board[0])
    tops = [top3_row(r) for r in board]
    pre = [[] for _ in range(m + 1)]
    for i in range(m):
        pre[i + 1] = merge_top3(pre[i], tops[i])
    suf = [[] for _ in range(m + 1)]
    for i in range(m - 1, -1, -1):
        suf[i] = merge_top3(suf[i + 1], tops[i])
    best = None
    for i in range(1, m - 1):
        P, S = pre[i], suf[i + 1]
        for c in range(n):
            v = board[i][c]
            for a, ca in P:
                if ca == c:
                    continue
                for b, cb in S:
                    if cb != c and cb != ca:
                        s = a + v + b
                        if best is None or s > best:
                            best = s
    return best

for b, exp in examples:
    print(prefix_suffix(b), '| kỳ vọng', exp)

4 | kỳ vọng 4
15 | kỳ vọng 15
3 | kỳ vọng 3


### 7.3. Cách 4 — Lọc ứng viên rồi tìm kiếm có cắt tỉa

Ứng viên = top-3 mỗi hàng **và** top-3 mỗi cột (áp dụng nhận xét 3.3 cho cả cột): mỗi quân của lời giải tối ưu có thể chọn trong top-3 của hàng nó; lặp lại lý luận cho cột ⇒ có lời giải tối ưu mà mọi quân nằm trong giao "top-3 hàng" (lọc lần đầu) rồi "top-3 cột trong tập đã lọc". Sắp ứng viên giảm dần và duyệt bộ ba với cắt tỉa theo cận trên.

In [7]:
def candidates_search(board):
    m, n = len(board), len(board[0])
    cand = [(v, i, j) for i, row in enumerate(board) for v, j in top3_row(row)]
    by_col = {}
    for v, i, j in cand:
        by_col.setdefault(j, []).append((v, i, j))
    cand = [x for lst in by_col.values() for x in sorted(lst, reverse=True)[:3]]
    cand.sort(reverse=True)
    best = None
    L = len(cand)
    for x in range(L):
        v1, r1, c1 = cand[x]
        if best is not None and x + 2 < L and v1 + cand[x + 1][0] + cand[x + 2][0] <= best:
            break
        for y in range(x + 1, L):
            v2, r2, c2 = cand[y]
            if r2 == r1 or c2 == c1:
                continue
            if best is not None and y + 1 < L and v1 + v2 + cand[y + 1][0] <= best:
                break
            for z in range(y + 1, L):
                v3, r3, c3 = cand[z]
                if r3 in (r1, r2) or c3 in (c1, c2):
                    continue
                s = v1 + v2 + v3
                if best is None or s > best:
                    best = s
                break                                         # ứng viên z đầu tiên hợp lệ là tốt nhất cho (x, y)
    return best

for b, exp in examples:
    print(candidates_search(b), '| kỳ vọng', exp)

4 | kỳ vọng 4
15 | kỳ vọng 15
3 | kỳ vọng 3


Lọc theo cột sau khi lọc theo hàng vẫn an toàn: áp dụng lại lý luận 3.3 với tập ứng viên (mỗi quân chỉ cần nằm trong top-3 của cột nó **trong tập ứng viên**).

## 8. TRACE từng bước

Trace cách 3 trên ví dụ 2: in `pre`, `suf` và bộ ba tốt nhất cho mỗi hàng giữa.

In [8]:
def trace(board):
    m, n = len(board), len(board[0])
    tops = [top3_row(r) for r in board]
    pre = [[] for _ in range(m + 1)]
    for i in range(m):
        pre[i + 1] = merge_top3(pre[i], tops[i])
    suf = [[] for _ in range(m + 1)]
    for i in range(m - 1, -1, -1):
        suf[i] = merge_top3(suf[i + 1], tops[i])
    for i in range(1, m - 1):
        best = None
        for c in range(n):
            for a, ca in pre[i]:
                for b, cb in suf[i + 1]:
                    if len({ca, cb, c}) == 3:
                        cand = (a + board[i][c] + b, (ca, c, cb))
                        best = cand if best is None or cand > best else best
        print(f"hàng giữa {i}: trên {pre[i]}, dưới {suf[i + 1]} -> tốt nhất {best}")
    print('đáp án:', prefix_suffix(board), '| vét cạn:', brute(board))

trace(examples[1][0])

hàng giữa 1: trên [(3, 2), (2, 1), (1, 0)], dưới [(9, 2), (8, 1), (7, 0)] -> tốt nhất (15, (2, 1, 0))
đáp án: 15 | vét cạn: 15


## 9. Code nộp LeetCode

Với `m ≤ 100`, duyệt bộ ba hàng là đủ; `break` sau ô hợp lệ đầu tiên của hàng thứ ba vì `tops[k]` đã sắp giảm dần.

In [9]:
class Solution:
    def maximumValueSum(self, board: List[List[int]]) -> int:
        m = len(board)
        tops = [sorted(((v, j) for j, v in enumerate(row)), reverse=True)[:3] for row in board]
        best = -float('inf')
        for i in range(m):
            ti = tops[i]
            for j in range(i + 1, m):
                tj = tops[j]
                for k in range(j + 1, m):
                    tk = tops[k]
                    for a, ca in ti:
                        for b, cb in tj:
                            if cb == ca:
                                continue
                            ab = a + b
                            for c, cc in tk:
                                if cc != ca and cc != cb:
                                    if ab + c > best:
                                        best = ab + c
                                    break          # tk đã sắp giảm dần: ô hợp lệ đầu tiên là tốt nhất
        return best

for b, exp in examples:
    got = Solution().maximumValueSum(b)
    print(got, 'OK' if got == exp else 'SAI')

4 OK
15 OK
3 OK


## 10. Vì sao thuật toán đúng

1. **Mọi lời giải có dạng (trên, giữa, dưới):** sắp ba quân theo hàng tăng ⇒ quân giữa ở hàng `i` với `1 ≤ i ≤ m − 2`, quân trên ở hàng `< i`, quân dưới ở hàng `> i`.
2. **Top-3 cột phân biệt là đủ cho quân trên:** cố định quân giữa và quân dưới (chiếm 2 cột). Gọi `X` là quân trên tối ưu. Nếu `X ∉ pre[i]`, trong `pre[i]` (3 cột phân biệt) có một ô ở cột khác 2 cột kia; giá trị của nó `≥ X` (vì `pre[i]` gồm các ô tốt nhất theo từng cột, rồi 3 cột tốt nhất) ⇒ thay được. Tương tự quân dưới với `suf[i + 1]`.
3. **Duyệt đầy đủ:** mọi cột của hàng giữa được thử ⇒ không bỏ sót.
4. **Giá trị âm:** khởi tạo `−∞`, không phải 0 (đáp án có thể âm).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(m³ n³)` | `O(1)` |
| 2. Top-3 + bộ ba hàng | `O(m n + m³ · 27)` | `O(m)` |
| 3. Tiền tố/hậu tố + hàng giữa | `O(m n · 9)` | `O(m)` |
| 4. Ứng viên + cắt tỉa | `O(mn + K³)` xấu nhất, thường nhanh | `O(m)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [10]:
impls = {'row_triples': row_triples, 'prefix_suffix': prefix_suffix, 'candidates': candidates_search,
         'submit': lambda b: Solution().maximumValueSum(b)}
edge = [([[-1] * 3] * 3, -3), ([[5, 0, 0], [5, 0, 0], [5, 0, 0]], 5),
        ([[10 ** 9] * 3] * 3, 3 * 10 ** 9), ([[1, 2, 3, 4]] * 3, 9), ([[9, 9, 1], [9, 9, 1], [1, 1, 1]], 19)]
for b, exp in examples + edge:
    assert brute(b) == exp, (b, brute(b))
    for name, f in impls.items():
        assert f(b) == exp, (name, b)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [11]:
import random
rng = random.Random(3256)
cnt = 0
for _ in range(300):
    m, n = rng.randint(3, 6), rng.randint(3, 6)
    V = rng.choice([3, 10, 10 ** 9])
    b = [[rng.randint(-V, V) for _ in range(n)] for _ in range(m)]
    exp = brute(b)
    for name, f in impls.items():
        assert f(b) == exp, (name, b)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bàn ngẫu nhiên (kể cả giá trị âm, nhiều giá trị trùng).')

Khớp vét cạn trên 300 bàn ngẫu nhiên (kể cả giá trị âm, nhiều giá trị trùng).


### 12.3. Ca lớn `100 × 100`

In [12]:
import time
big = {
    'ngẫu nhiên ±1e9': [[rng.randint(-10 ** 9, 10 ** 9) for _ in range(100)] for _ in range(100)],
    'một cột rất lớn': [[10 ** 9 if j == 7 else rng.randint(-5, 5) for j in range(100)] for _ in range(100)],
    'toàn âm': [[-rng.randint(1, 10 ** 9) for _ in range(100)] for _ in range(100)],
}
for name, b in big.items():
    t = time.time(); a = Solution().maximumValueSum(b); t1 = time.time() - t
    t = time.time(); c = prefix_suffix(b); t2 = time.time() - t
    assert a == c == candidates_search(b)
    print(f"{name:18s}: {a:12d} | nộp (bộ ba hàng) {t1:.2f}s | tiền tố/hậu tố {t2:.2f}s")

ngẫu nhiên ±1e9   :   2998537855 | nộp (bộ ba hàng) 0.23s | tiền tố/hậu tố 0.02s


một cột rất lớn   :   1000000010 | nộp (bộ ba hàng) 0.29s | tiền tố/hậu tố 0.01s


toàn âm           :      -281531 | nộp (bộ ba hàng) 0.21s | tiền tố/hậu tố 0.01s


Kiểm tra độc lập: trên bàn `60 × 60` (đủ nhỏ cho cách 2 — top-3 + mọi bộ ba hàng), ba cách cho cùng kết quả.

In [13]:
for _ in range(3):
    b = [[rng.randint(-10 ** 9, 10 ** 9) for _ in range(60)] for _ in range(60)]
    assert row_triples(b) == prefix_suffix(b) == Solution().maximumValueSum(b)
b = [[rng.randint(0, 3) for _ in range(60)] for _ in range(60)]
assert row_triples(b) == Solution().maximumValueSum(b) == candidates_search(b)
print('60 × 60: cách 2 = cách 3 = cách 4 = lời giải nộp.')

60 × 60: cách 2 = cách 3 = cách 4 = lời giải nộp.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam: mỗi hàng lấy ô lớn nhất, chọn 3 hàng lớn nhất

In [14]:
def wrong_greedy(board):
    best_rows = sorted(((max(r), r.index(max(r))) for r in board), reverse=True)[:3]
    return sum(v for v, _ in best_rows)                        # SAI: có thể trùng cột

bad = [[5, 0, 0], [5, 0, 0], [5, 0, 0]]
print('sai :', wrong_greedy(bad), '| đúng:', Solution().maximumValueSum(bad))

sai : 15 | đúng: 5


**Vì sao sai:** ba số `5` cùng cột — chỉ được dùng một.

### Lỗi 2 — Chỉ giữ top-**2** mỗi hàng

In [15]:
def wrong_top2(board):
    m = len(board)
    tops = [sorted(((v, j) for j, v in enumerate(r)), reverse=True)[:2] for r in board]   # SAI
    best = None
    for i, j, k in combinations(range(m), 3):
        for a, ca in tops[i]:
            for b, cb in tops[j]:
                for c, cc in tops[k]:
                    if len({ca, cb, cc}) == 3:
                        s = a + b + c
                        best = s if best is None or s > best else best
    return best

bad = [[9, 9, 1], [9, 9, 1], [9, 9, 1]]
print('sai :', wrong_top2(bad), '| đúng:', Solution().maximumValueSum(bad))

sai : None | đúng: 19


**Vì sao sai:** ba hàng giống nhau với hai ô lớn ở cùng hai cột: quân thứ ba **buộc** phải dùng ô thứ ba (`1`) — top-2 không chứa nó nên không tìm được lời giải (`None`).

### Lỗi 3 — Gộp tiền tố top-3 **không** khử trùng cột

In [16]:
def wrong_merge_no_dedup(board):
    m, n = len(board), len(board[0])
    tops = [top3_row(r) for r in board]
    pre = [[] for _ in range(m + 1)]
    for i in range(m):
        pre[i + 1] = sorted(pre[i] + tops[i], reverse=True)[:3]        # SAI: có thể 3 ô cùng cột
    suf = [[] for _ in range(m + 1)]
    for i in range(m - 1, -1, -1):
        suf[i] = sorted(suf[i + 1] + tops[i], reverse=True)[:3]
    best = None
    for i in range(1, m - 1):
        for c in range(n):
            for a, ca in pre[i]:
                for b, cb in suf[i + 1]:
                    if len({ca, cb, c}) == 3:
                        s = a + board[i][c] + b
                        best = s if best is None or s > best else best
    return best

found = None
r_ = random.Random(6)
for _ in range(5000):
    b = [[r_.randint(0, 4) for _ in range(3)] for _ in range(r_.randint(3, 5))]
    if wrong_merge_no_dedup(b) != brute(b):
        found = b; break
print('phản ví dụ:', found, '-> sai:', wrong_merge_no_dedup(found), '| đúng:', Solution().maximumValueSum(found))

phản ví dụ: [[0, 0, 3], [3, 4, 0], [1, 2, 2], [0, 0, 2], [2, 2, 3]] -> sai: 8 | đúng: 9


**Vì sao sai:** phần trên có thể có 3 ô tốt nhất **cùng một cột** (ở các hàng khác nhau); nếu cột đó bị chiếm thì không còn lựa chọn thay thế.

### Lỗi 4 — Khởi tạo đáp án bằng 0

In [17]:
def wrong_init_zero(board):
    r = Solution().maximumValueSum(board)
    return max(0, r)                                           # SAI: tương đương khởi tạo best = 0

print('sai :', wrong_init_zero([[-1] * 3] * 3), '| đúng:', Solution().maximumValueSum([[-1] * 3] * 3))

sai : 0 | đúng: -3


**Vì sao sai:** **bắt buộc** đặt đủ ba quân; với bàn toàn số âm đáp án âm.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chọn `k` ô không chung hàng/cột, `k` nhỏ" ⇒ mỗi ô chỉ cần nằm trong top-`k` của hàng (và cột) của nó.
- Cố định phần tử "giữa", tiền tố/hậu tố cho phần trước/sau.
- Bài toán tổng quát (chọn `m` ô hoán vị) là bài **phân công** (Hungarian) — với `k = 3` không cần.

## 15. Biến thể của bài

1. **LC 3257 (bản II):** `m, n ≤ 500` — tiền tố/hậu tố top-3 + hàng giữa.
2. Hai quân xe ⇒ top-2 mỗi hàng, tiền tố top-2.
3. Bốn quân xe ⇒ top-4, cố định hai hàng giữa? Nhanh nhất dùng luồng chi phí với `k = 4` đơn vị.
4. Quân xe **có thể** chung cột nhưng không chung hàng ⇒ chỉ cần top-1 mỗi hàng.

In [18]:
# Biến thể 2: hai quân xe
def two_rooks(board):
    m = len(board)
    tops = [sorted(((v, j) for j, v in enumerate(r)), reverse=True)[:2] for r in board]
    pre = []
    best = None
    for i in range(m):
        for a, ca in pre:
            for b, cb in tops[i]:
                if ca != cb:
                    best = a + b if best is None or a + b > best else best
        pre = merge_top3(pre, tops[i])[:2]
    return best

def two_brute(board):
    m, n = len(board), len(board[0])
    return max(board[i][a] + board[j][b] for i in range(m) for j in range(i + 1, m) for a in range(n) for b in range(n) if a != b)

rng = random.Random(15)
for _ in range(300):
    b = [[rng.randint(-5, 5) for _ in range(rng.randint(2, 5))] for _ in range(rng.randint(2, 5))]
    n = min(len(r) for r in b); b = [r[:n] for r in b]
    if n < 2:
        continue
    assert two_rooks(b) == two_brute(b)
print('Biến thể hai quân xe (top-2 + tiền tố top-2) khớp vét cạn.')

Biến thể hai quân xe (top-2 + tiền tố top-2) khớp vét cạn.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Vét cạn | `O(m³n³)` | Oracle |
| 2 | Top-3 + bộ ba hàng | `O(m³)` | **Nộp bài** (bản I) |
| 3 | Tiền tố/hậu tố + hàng giữa | `O(9mn)` | Bắt buộc cho bản II |
| 4 | Ứng viên + cắt tỉa | thường rất nhanh | Phụ thuộc dữ liệu |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `[[1,0,0],[0,1,0],[0,0,1]]`.

**Bài 2 (tính tay):** `[[3,3,3],[3,3,3],[0,0,9]]`.

**Bài 3 (điền TODO):** trả về luôn **vị trí** ba quân xe.

**Bài 4 (tự viết lại):** biến thể 4 — được chung cột, không chung hàng.

**Bài 5 (mini challenge):** bốn quân xe trên bàn `≤ 30 × 30` (top-4 + duyệt có cắt tỉa).

In [19]:
def positions_todo(board):
    # TODO
    raise NotImplementedError

def shared_columns_todo(board):
    # TODO
    raise NotImplementedError

def four_rooks_todo(board):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Đường chéo. (2) `1 + 1 + 1`. (3) `3`.

**Bài 2.** (1) Hàng 2 lấy `9` (cột 2). (2) Hai hàng trên lấy `3` ở cột 0, 1. (3) `15`.

**Bài 3.** (1) Lưu `(a, ca, ra)` trong tiền tố (thêm hàng). (2) Cập nhật cùng lúc với max. (3) Kiểm hợp lệ.

**Bài 4.** (1) Chỉ cần ba hàng khác nhau. (2) Top-1 mỗi hàng, lấy 3 hàng lớn nhất. (3) Đó chính là "lỗi 1" — đúng cho biến thể này!

**Bài 5.** (1) Top-4 mỗi hàng. (2) Lọc theo cột top-4. (3) Duyệt bộ bốn có cắt tỉa.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2, 4
assert Solution().maximumValueSum([[1, 0, 0], [0, 1, 0], [0, 0, 1]]) == 3
assert Solution().maximumValueSum([[3, 3, 3], [3, 3, 3], [0, 0, 9]]) == 15

def shared_columns(board):
    return sum(sorted((max(r) for r in board), reverse=True)[:3])

rng = random.Random(3)
for _ in range(200):
    b = [[rng.randint(-5, 5) for _ in range(3)] for _ in range(rng.randint(3, 5))]
    exp = max(b[i][x] + b[j][y] + b[k][z] for i, j, k in combinations(range(len(b)), 3)
              for x in range(3) for y in range(3) for z in range(3))
    assert shared_columns(b) == exp
print('Bài 1: 3 | Bài 2: 15 | Bài 4 khớp vét cạn.')

Bài 1: 3 | Bài 2: 15 | Bài 4 khớp vét cạn.


In [21]:
# Bài 3, 5
def positions(board):
    m, n = len(board), len(board[0])
    tops = [[(v, j, i) for v, j in top3_row(r)] for i, r in enumerate(board)]
    def merge(cur, cand):
        best = {}
        for v, j, i in cur + cand:
            if j not in best or v > best[j][0]:
                best[j] = (v, j, i)
        return sorted(best.values(), reverse=True)[:3]
    pre = [[] for _ in range(m + 1)]
    for i in range(m):
        pre[i + 1] = merge(pre[i], tops[i])
    suf = [[] for _ in range(m + 1)]
    for i in range(m - 1, -1, -1):
        suf[i] = merge(suf[i + 1], tops[i])
    best = None
    for i in range(1, m - 1):
        for c in range(n):
            for a, ca, ra in pre[i]:
                for b, cb, rb in suf[i + 1]:
                    if len({ca, cb, c}) == 3:
                        cand = (a + board[i][c] + b, [(ra, ca), (i, c), (rb, cb)])
                        if best is None or cand[0] > best[0]:
                            best = cand
    return best

def four_rooks(board):
    m, n = len(board), len(board[0])
    cand = [(v, i, j) for i, row in enumerate(board) for v, j in sorted(((v, j) for j, v in enumerate(row)), reverse=True)[:4]]
    by_col = {}
    for v, i, j in cand:
        by_col.setdefault(j, []).append((v, i, j))
    cand = sorted((x for lst in by_col.values() for x in sorted(lst, reverse=True)[:4]), reverse=True)
    best = None
    def go(start, used_r, used_c, total, depth):
        nonlocal best
        if depth == 4:
            best = total if best is None or total > best else best
            return
        for t in range(start, len(cand)):
            v, i, j = cand[t]
            if best is not None and total + v * (4 - depth) <= best:
                return
            if i in used_r or j in used_c:
                continue
            go(t + 1, used_r | {i}, used_c | {j}, total + v, depth + 1)
    go(0, frozenset(), frozenset(), 0, 0)
    return best

rng = random.Random(4)
for _ in range(100):
    b = [[rng.randint(-9, 9) for _ in range(rng.randint(4, 5))] for _ in range(rng.randint(4, 5))]
    n = min(map(len, b)); b = [r[:n] for r in b]
    val, pos = positions(b)
    assert val == brute(b) and len({r for r, c in pos}) == 3 and len({c for r, c in pos}) == 3
    assert sum(b[r][c] for r, c in pos) == val
    exp4 = max(sum(b[r][c] for r, c in zip(rows, cols)) for rows in combinations(range(len(b)), 4) for cols in permutations(range(n), 4))
    assert four_rooks(b) == exp4
print('Bài 3 (vị trí) và Bài 5 (bốn quân xe) khớp vét cạn. Ví dụ 1:', positions(examples[0][0]))

Bài 3 (vị trí) và Bài 5 (bốn quân xe) khớp vét cạn. Ví dụ 1: (4, [(0, 2), (1, 3), (2, 1)])


Lưu ý bài 5: cắt tỉa `total + v·(4 − depth) ≤ best` hợp lệ vì ứng viên đã sắp giảm dần (mọi quân sau có giá trị `≤ v`).

## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
tops[i] = top-3 (v, cột) của hàng i
pre[i+1] = merge_top3(pre[i], tops[i])   # mỗi cột giữ 1 ô tốt nhất, lấy 3
suf[i]   = merge_top3(suf[i+1], tops[i])
for i in 1..m-2: for c: for (a, ca) ∈ pre[i]: for (b, cb) ∈ suf[i+1]: nếu ba cột khác nhau: cập nhật
```

**Checklist:**

- [ ] Top-**3** (không phải top-1/top-2)?
- [ ] Gộp tiền tố/hậu tố khử trùng **cột**?
- [ ] Ba cột đôi một khác nhau?
- [ ] Khởi tạo `−∞` (giá trị âm)?
- [ ] Hàng giữa chạy `1..m−2`?